# Imports

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Helpers

In [ ]:
def softmax(s, a, theta):
    phi = np.zeros(32)
    phi[a*8:a*8+8] = s
    numerator = np.exp(phi.T @ theta)
    
    denominator = 0
    for a2 in range(4):
        phi2 = np.zeros(32)
        phi2[a2*8:a2*8+8] = s
        denominator += np.exp(phi2.T @ theta)
    
    pi = numerator / denominator
    return pi

In [ ]:
def gradient(s, a, theta):
    phi = np.zeros(32)
    phi[a*8:a*8+8] = s

    expected = np.zeros(32)
    for a2 in range(4):
        phi2 = np.zeros(32)
        phi2[a2*8:a2*8+8] = s
        expected += phi2 * softmax(s, a2, theta)
    
    return phi - expected

In [ ]:
def compute_returns(rewards, gamma):
    returns = []
    v = 0
    for r in reversed(rewards):
        v = r + gamma * v
        returns.insert(0, v)
    return returns

Main Reinforce Function

In [ ]:
def reinforce(env, theta, alpha, gamma, n_episodes):
    for episode in range(n_episodes):
        
        states, actions, rewards = [], [], []
        
        # env.reset() returns (state, info) 
        state, info = env.reset()
        done = False
        
        # generate full episode
        while not done:
            # sample action:
            probs = np.array([softmax(state, a, theta) for a in range(4)])
            action = np.random.choice(4, p=probs)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            # store state, action, reward in lists
            rewards.append(reward)
            states.append(state)
            actions.append(action)
            state = next_state
        returns = compute_returns(rewards, gamma)
        for t in range(len(states)):
            theta += alpha * gradient(s, a, theta) * compute_returns(rewards, gamma)
    return theta